# Neuromorphic Fault Detection — Training Walkthrough (Trial 1)

Cell-by-cell walkthrough of the event-driven SNN fault-detection pipeline: build the CWRU dataset, delta-encode it into spikes, train the `LIFClassifier`, then evaluate with the incipient-fault severity-tier breakdown and the energy/sparsity comparison against a dense CNN1D baseline.

Run cells top to bottom. Kernel: **Neuromorphic Fault Detection (.venv)**. The training cell's `epochs` variable is separate so you can try a quick run first.

In [ ]:
%matplotlib inline

## Setup

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from utils.config import Config
from utils.seed import set_seed
from utils.device import get_device

from datasets.cwru import CWRUDataset
from datasets.labels import get_class_names
from encoding.factory import create_encoder
from encoding.visualization import SpikeVisualizer

from train import build_spike_transform, compute_or_load_split
from models.factory import build_model

from training.loss import build_loss
from training.optimizer import build_optimizer
from training.early_stopping import EarlyStopping
from training.checkpoint import CheckpointManager
from training.trainer import Trainer

from evaluation.metrics import (
    classification_metrics,
    confidence_margin,
    softmax_confidence,
    per_class_breakdown,
    severity_tier_breakdown,
    fault_severity_matrix,
)
from explainability.spike_activity import SpikeActivityAnalyzer

import evaluate  # reuse its baseline-training / energy-comparison helpers

In [ ]:
# `evaluation.report` (pulled in transitively above) forces the headless
# "Agg" matplotlib backend on import, since evaluate.py only ever saves
# figures to disk. Restore inline plotting for this notebook.
%matplotlib inline

In [ ]:
cfg = Config("configs/default.yaml").data
set_seed(cfg["seed"])
device = get_device()

print("device:", device)
cfg

## 1. Dataset

Load the CWRU windows and delta-encode them into ON/OFF spike trains.

In [ ]:
transform = build_spike_transform(cfg)

dataset = CWRUDataset(
    root=cfg["dataset"]["root"],
    window_size=cfg["dataset"]["window_size"],
    overlap=cfg["dataset"]["overlap"],
    channel=cfg["dataset"]["channel"],
    transform=transform,
)

dataset.summary()

### Inspect one raw sample and its delta-encoded spikes

In [ ]:
raw_dataset = CWRUDataset(
    root=cfg["dataset"]["root"],
    window_size=cfg["dataset"]["window_size"],
    overlap=cfg["dataset"]["overlap"],
    channel=cfg["dataset"]["channel"],
)

raw_signal, label = raw_dataset[0]
print("raw signal shape:", raw_signal.shape, "label:", label)

encoder = create_encoder("delta", threshold=cfg["encoding"]["threshold"])
spikes = encoder.encode(raw_signal)

fig, ax = SpikeVisualizer.delta_events(raw_signal[:500], spikes[:, :500])
plt.show()

print("spike sparsity (fraction inactive):", 1 - spikes.mean())

## 2. Train / validation / test split

In [ ]:
results_dir = Path(cfg["results"]["output_dir"])

train_idx, val_idx, test_idx = compute_or_load_split(dataset, cfg, results_dir)
print(f"train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}")

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=cfg["training"]["batch_size"], shuffle=True)
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=cfg["training"]["batch_size"], shuffle=False)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=cfg["training"]["batch_size"], shuffle=False)

## 3. Build the model

In [ ]:
model = build_model(cfg)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\ntotal parameters: {n_params:,}")

### Sanity forward pass

In [ ]:
sample_x, sample_y = next(iter(train_loader))
sample_out = model(sample_x, return_all=True)

print("input shape:", sample_x.shape)
print("output spikes shape:", sample_out["spikes"].shape)
print("hidden layer shapes:", [h.shape for h in sample_out["hidden_spikes"]])

## 4. Training setup

In [ ]:
output_dir = results_dir / cfg["model"]["architecture"]

loss_fn = build_loss(cfg["training"]["loss"])
optimizer = build_optimizer(cfg["training"]["optimizer"].lower(), model.parameters(), lr=cfg["training"]["lr"])
early_stopping = EarlyStopping(**cfg["training"]["early_stopping"])
checkpoint_manager = CheckpointManager(output_dir / "checkpoints", metric_name="val_accuracy", mode="max")

trainer = Trainer(
    model,
    optimizer,
    loss_fn,
    device,
    checkpoint_manager=checkpoint_manager,
    early_stopping=early_stopping,
    output_dir=output_dir,
)

## 5. Train

Edit `epochs` below to try a quick run first (e.g. 10–15) before committing to the full config value (real data is much bigger than what this pipeline was smoke-tested on, so per-epoch time on CPU is untested — start small).

In [ ]:
epochs = cfg["training"]["epochs"]  # <- edit this for a quicker first pass

history = trainer.fit(train_loader, val_loader, epochs=epochs)

### Training curve

In [ ]:
epochs_ran = [h["epoch"] for h in history]
train_acc = [h["train"]["accuracy"] for h in history]
val_acc = [h["val"]["accuracy"] for h in history]
train_loss = [h["train"]["loss"] for h in history]
val_loss = [h["val"]["loss"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_ran, train_loss, label="train")
axes[0].plot(epochs_ran, val_loss, label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs_ran, train_acc, label="train")
axes[1].plot(epochs_ran, val_acc, label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate on the held-out test set

Reloads the best checkpoint saved during training above.

In [ ]:
checkpoint_manager.load_best(model, device=device)

result = trainer.evaluate(test_loader, return_all=True)

time_steps = cfg["dataset"]["window_size"] // cfg["encoding"]["bin_size"]
margin = confidence_margin(result["logits"], time_steps)
conf = softmax_confidence(result["logits"])

metrics = classification_metrics(result["y_true"], result["y_pred"])
metrics

### Severity-tier breakdown (incipient fault detection — the core result)

In [ ]:
per_class = per_class_breakdown(result["y_true"], result["y_pred"], margin, conf)
per_class

In [ ]:
severity = severity_tier_breakdown(
    result["y_true"], result["y_pred"], margin,
    confidence_threshold=cfg["evaluation"]["confidence_threshold"],
)
severity

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(severity["severity_tier"], severity["accuracy"])
ax.set_title("Accuracy by fault severity (incipient detection)")
ax.set_xlabel("severity tier")
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05)
plt.show()

### Fault-type × severity heatmap

In [ ]:
matrix = fault_severity_matrix(result["y_true"], result["y_pred"], margin=margin)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title("Accuracy: fault type x severity")
plt.show()

### Confusion matrix

In [ ]:
class_names = get_class_names()
cm = confusion_matrix(result["y_true"], result["y_pred"], labels=list(range(len(class_names))))

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. (Optional) Energy/sparsity comparison vs. dense CNN1D baseline

This trains a second model (CNN1D) from scratch on the same split for a fair comparison, so it takes roughly as long as the training cell above. Skip if you just want the SNN results for now.

In [ ]:
baseline_result = evaluate._train_and_evaluate_baseline(cfg, results_dir, device)
baseline_result["metrics"]

In [ ]:
sample_x_snn, _ = next(iter(test_loader))
sample_x_snn = sample_x_snn.to(device)
model.eval()
with torch.no_grad():
    sample_output_snn = model(sample_x_snn, return_all=True)

snn_result_for_energy = {
    "model": model,
    "metrics": metrics,
    "sample_input": sample_x_snn,
    "sample_output": sample_output_snn,
}

energy_table = evaluate._energy_comparison_table(cfg, snn_result_for_energy, baseline_result)
energy_table

## 8. Spike activity / interpretability

In [ ]:
analyzer = SpikeActivityAnalyzer(model, device)
records = analyzer.collect(test_loader, max_batches=5)

print("layer sparsity:", analyzer.layer_sparsity(records))

In [ ]:
profile = analyzer.class_spike_profile(records, num_classes=cfg["model"]["num_classes"])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(profile.numpy(), annot=True, fmt=".1f", cmap="viridis", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("output neuron (spike count)")
ax.set_ylabel("true class")
ax.set_title("Mean output spike-count profile per true class")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = analyzer.plot_sample_raster(records, layer="output")
plt.show()

## Notes

- Checkpoints: `results/lif_classifier/checkpoints/` (and `results/cnn1d/checkpoints/` if you ran the baseline cell).
- The train/val/test split is cached at `results/splits.json`, so re-running this notebook (or `train.py` / `evaluate.py`) reuses the identical held-out test set — delete that file if you want a fresh split.
- To run the full scripted pipeline non-interactively instead, use `python train.py` and `python evaluate.py` from the terminal.
- Next milestone (Phase B): real Arduino/TFLite-Micro deployment and measured energy, plus STDP-based online adaptation.